# Paper figure additions from the completed 08 series

This notebook only redraws saved results. It does not train models, change labels, select cutoffs, or edit any manuscript source. Run all cells in the existing project environment in Antigravity.

The Overleaf ZIP and all 82 original archive members remain unchanged. New PDF and PNG figures, source-table copies, and provenance are added under `overleaf_revision_08_series_20261001/revision_figures_08`. The new files are not inserted into `paper.tex`; paragraph-by-paragraph editing will determine their final placement and captions.

Primary figure summaries give fireline units equal weight. This reporting choice does not change the executed condition-tree training weights, inner-validation objectives, or SHAP averaging. Equal-fire/source sensitivities remain explicitly identified. The 08D intervals use 2,000 whole-fire bootstrap resamples, not the 5,000 used for several earlier analyses.


In [ ]:
from pathlib import Path
import json, hashlib, shutil
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/"outputs/08_condition_outcomes").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run from the wildfire_fireline_effectiveness project or its notebooks folder.")
PROJECT = ROOT/"overleaf_revision_08_series_20261001"
OUT = PROJECT/"revision_figures_08"
RUN = ROOT/"outputs/08_condition_outcomes/08D_20261001T002458_892164Z_f3e697"
if not (PROJECT/"paper.tex").exists():
    raise RuntimeError("The unchanged Overleaf working copy is required.")
assert OUT.resolve().is_relative_to(PROJECT.resolve())
OUT.mkdir(exist_ok=True)
(OUT/"source_tables").mkdir(exist_ok=True)
original_paper_hash = hashlib.sha256((PROJECT/"paper.tex").read_bytes()).hexdigest()
sources = {
 "baseline_error_comparisons.csv": RUN/"baseline_error_comparisons.csv",
 "frozen_rule_weighting_checks.csv": RUN/"frozen_rule_weighting_checks.csv",
 "cutoff_common_fire_sensitivity.csv": RUN/"cutoff_common_fire_sensitivity.csv",
 "forward_year_metrics.csv": PROJECT/"data_tables/TABLE_5_FORWARD_TEMPORAL_GENERALIZATION_V1.csv",
}
tables = {name: pd.read_csv(path) for name,path in sources.items()}
for name,path in sources.items():
    shutil.copy2(path, OUT/"source_tables"/name)
baseline=tables["baseline_error_comparisons.csv"]
groups=tables["frozen_rule_weighting_checks.csv"]
cutoffs=tables["cutoff_common_fire_sensitivity.csv"]
temporal=tables["forward_year_metrics.csv"]
for df in [baseline,groups,cutoffs]:
    df["scope"]=df["scope"].astype(str)
base=baseline.loc[(baseline.scheme=="forward_year")&(baseline.model=="LightGBM")&
                 (baseline.metric=="brier")&(baseline.evaluation_weight=="unit")&
                 (baseline.baseline_weight=="unit")].copy()
assert base.scope.is_unique
combined=base.set_index("scope").loc["all_test_years"]
assert np.isclose(combined.model_error,0.1924647278,atol=1e-9,rtol=0)
assert np.isclose(combined.baseline_error,0.2223587386,atol=1e-9,rtol=0)
order=[3,4,6,7,10,11,13,14]
g=groups.loc[(groups.scope=="2023_2024")&(groups.weight=="unit")].set_index("leaf_id").loc[order]
assert len(g)==8 and g.n_units.sum()==122798
assert np.allclose(g.held_fraction,g.n_held/g.n_units)
assert (g.ci_lower<=g.held_fraction).all() and (g.ci_upper>=g.held_fraction).all()
print("Loaded saved results. No model fitting or new statistical selection.")


In [ ]:
plt.rcParams.update({
 "font.family":"DejaVu Sans","font.size":10,"axes.titlesize":11,"axes.labelsize":10,
 "xtick.labelsize":10,"ytick.labelsize":10,"legend.fontsize":10,
 "pdf.fonttype":42,"ps.fonttype":42,"savefig.facecolor":"white",
 "axes.spines.top":False,"axes.spines.right":False,
})
BLUE="#0072B2"; ORANGE="#D55E00"; GRAY="#666666"; GREEN="#009E73"
generated=[]
def clean(ax,axis="y"):
    ax.set_axisbelow(True)
    ax.grid(axis=axis,color="#e5e5e5",linewidth=.6)
def save(fig,name):
    for ext in ["pdf","png"]:
        path=OUT/(name+"."+ext)
        fig.savefig(path,dpi=300,bbox_inches="tight",pad_inches=.10)
        generated.append(path)
    plt.close(fig)
    display(Image(filename=str(OUT/(name+".png"))))
print("Style ready: vector PDF and 300-dpi PNG exports.")


## Main-paper candidate: forward-year performance

Replace the older equal-fire Brier panel only when its paragraph, caption, and table have been revised together. Within-fire ROC-AUC remains a separate ranking question. Annual curves are point estimates. The 2021 pooled Brier difference has a bootstrap interval crossing zero; do not claim independently established improvement in every year.


In [ ]:
years=[2021,2022,2023,2024]
t=temporal.loc[temporal.model_id=="FORWARD_FIXED_05C"].set_index("test_year").loc[years]
b=base.set_index("scope").loc[list(map(str,years))]
fig,axes=plt.subplots(1,2,figsize=(8.0,3.65))
a=axes[0]
a.plot(years,t.pooled_roc_auc,"o-",color=BLUE,label="Pooled ROC-AUC",lw=1.6,ms=5)
a.plot(years,t.mixed_incident_macro_roc_auc,"s--",color=ORANGE,label="Within-fire ROC-AUC",lw=1.6,ms=5)
a.axhline(.5,color=GRAY,lw=1,ls=":",label="Chance ordering")
a.set(xlabel="Test year",ylabel="ROC-AUC (higher is better)",ylim=(.4,.8))
a.set_title("(a) Ordering Held above Burned Over",loc="left")
a.legend(loc="upper right",frameon=False,fontsize=9)
clean(a)
a=axes[1]
a.plot(years,b.baseline_error,"s--",color=GRAY,label="Historical Held-rate baseline",lw=1.6,ms=5)
a.plot(years,b.model_error,"o-",color=BLUE,label="LightGBM",lw=1.6,ms=5)
a.set(xlabel="Test year",ylabel="Pooled Brier score (lower is better)",ylim=(0,.3))
a.set_title("(b) Equal fireline-unit probability error",loc="left")
a.legend(loc="lower left",frameon=False,fontsize=9)
clean(a)
for a in axes:
    a.set_xticks(years)
    a.set_xlim(2020.8,2024.2)
fig.tight_layout(w_pad=2)
save(fig,"forward_year_unit_performance")


## Main-paper candidate: all eight fixed condition groups

The depth-three condition tree was fitted to recorded outcomes using 2020–2022 data with equal-fire/equal-source-record training weights. The tree is not the earlier LightGBM surrogate. The figure reports observed, unit-weighted Held fractions in 2023–2024. All eight groups are retained in training-tree order; no test-selected winner is shown. Source-record and fire counts can overlap between groups; fireline units do not.

Cutoffs are rounded for display only. RH is mean forecast relative humidity over 0–18 hours after prediction time. Gust is forecast gust at +18 hours. D is the minimum distance from the whole fireline-unit geometry to the mapped fire-area polygon at prediction time. D does not mean the entire fireline unit lies within that distance. E is mean elevation. A dozer type means the recorded Completed Dozer Line category; other types are not all hand-built firelines.


In [ ]:
labels=[
 "Other recorded types\nRH ≤ 35.6%; gust ≤ 7.81 m/s",
 "Other recorded types\nRH ≤ 35.6%; gust > 7.81 m/s",
 "Other recorded types\nRH > 35.6%; D ≤ 211 m",
 "Other recorded types\nRH > 35.6%; D > 211 m",
 "Completed Dozer Line\nD ≤ 10.4 m",
 "Completed Dozer Line\n10.4 m < D ≤ 150 m",
 "Completed Dozer Line\nD > 150 m; E ≤ 1,075 m",
 "Completed Dozer Line\nD > 150 m; E > 1,075 m",
]
labels=[f"{j+1}. {label}" for j,label in enumerate(labels)]
fig=plt.figure(figsize=(9.0,6.7))
ax=fig.add_axes([.40,.23,.40,.64])
y=np.arange(8)
for j,(_,r) in enumerate(g.iterrows()):
    color=BLUE if j<4 else ORANGE
    ax.errorbar(100*r.held_fraction,j,
       xerr=[[100*(r.held_fraction-r.ci_lower)],[100*(r.ci_upper-r.held_fraction)]],
       fmt="o",color=color,ecolor=color,capsize=3,ms=5,lw=1.5)
    ax.text(100*r.held_fraction,j-.27,f"{100*r.held_fraction:.1f}%",ha="center",va="center",fontsize=10)
ax.set_yticks(y,labels)
ax.tick_params(axis="y",length=0,pad=10)
ax.set(xlim=(0,102),ylim=(7.65,-.75),xlabel="Recorded Held fraction")
ax.xaxis.set_major_formatter(PercentFormatter(100,decimals=0))
ax.set_xticks([0,25,50,75,100])
ax.axvline(50,color=GRAY,ls=":",lw=1)
ax.axhline(3.5,color="#aaaaaa",lw=.7)
clean(ax,"x")
counts=fig.add_axes([.82,.23,.17,.64],sharey=ax);counts.axis("off")
counts.set_xlim(0,1)
for j,(_,r) in enumerate(g.iterrows()):
    counts.text(.5,j,f"{int(r.n_units):,} / {int(r.n_fires)}",ha="center",va="center",fontsize=10)
fig.text(.4,.92,"Held frequencies in eight condition groups",fontsize=13,weight="bold",ha="center")
fig.text(.02,.884,"Recorded type and conditions",fontsize=10,weight="bold")
fig.text(.905,.884,"Fireline units / fires",ha="center",fontsize=10,weight="bold")
fig.text(.02,.15,"Groups learned from 2020–2022; recorded outcomes evaluated in 2023–2024.",fontsize=10)
fig.text(.02,.116,"RH: forecast mean relative humidity, 0–18 h; gust: forecast at +18 h; E: mean elevation.",fontsize=10)
fig.text(.02,.082,"D: closest mapped fire-area-to-fireline distance. Other types exclude Completed Dozer Line.",fontsize=10)
fig.text(.02,.048,"Each fireline unit has equal weight. Bars: pointwise 95% intervals from 2,000 fire resamples.",fontsize=10)
save(fig,"condition_groups_unit_held_rates")


## Supplement candidate: both test years separately

Group identifiers refer to the same eight groups, in the same order as the main candidate. Open points identify groups below the fixed minimum test support (200 fireline units, 50 source records, 10 fires). The near-dozer group in 2023 has 133 fireline units. Intervals are pointwise, not simultaneous; these checks are exploratory rather than independent replication after the 08-series development.


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(9,4.8),sharex=True,sharey=True)
for ax,scope,title in zip(axes,["2023","2024","2023_2024"],["2023","2024","2023–2024 combined"]):
    d=groups.loc[(groups.scope==scope)&(groups.weight=="unit")].set_index("leaf_id").loc[order]
    for j,(_,r) in enumerate(d.iterrows()):
        color=BLUE if j<4 else ORANGE
        ax.errorbar(100*r.held_fraction,j,
          xerr=[[100*(r.held_fraction-r.ci_lower)],[100*(r.ci_upper-r.held_fraction)]],
          fmt="o",color=color,mfc=color if bool(r.support_pass) else "white",mec=color,
          capsize=2,ms=5,lw=1.2)
        ax.text(3,j,f"n = {int(r.n_units):,}",va="center",ha="left",fontsize=9)
    ax.set_title(title)
    ax.set(xlim=(0,103),ylim=(7.6,-.6),xlabel="Recorded Held fraction")
    ax.set_xticks([0,25,50,75,100])
    ax.xaxis.set_major_formatter(PercentFormatter(100,decimals=0))
    ax.axvline(50,color=GRAY,ls=":",lw=1)
    ax.axhline(3.5,color="#cccccc",lw=.7)
    clean(ax,"x")
axes[0].set_yticks(range(8),[f"Group {j}" for j in range(1,9)])
fig.suptitle("The same condition groups, evaluated in each test year",fontsize=13,y=.98)
fig.tight_layout(rect=[0,.13,1,.92])
fig.text(.02,.075,"n: fireline units. Group order matches the main condition figure; all points use equal-unit weighting.",fontsize=10)
fig.text(.02,.03,"Open point: below the minimum test support. Bars: pointwise 95% whole-fire bootstrap intervals.",fontsize=10)
save(fig,"supp_condition_groups_by_year")


## Supplement candidate: averaging and historical baselines

Saved forward-year LightGBM predictions are unchanged. Each row uses matching weights to calculate the training-only constant and test error. The source-record rows allocate a total weight of one to all retained pieces from a source record, rather than treating the record as a complete constructed fireline. The two equal-fire rows additionally give every fire equal total weight. Negative differences favor LightGBM.


In [ ]:
w=["unit","source","fire_unit","fire_source"]
d=baseline.loc[(baseline.scheme=="forward_year")&(baseline.scope=="all_test_years")&
 (baseline.model=="LightGBM")&(baseline.metric=="brier")&
 (baseline.evaluation_weight==baseline.baseline_weight)].set_index("evaluation_weight").loc[w]
fig,ax=plt.subplots(figsize=(8.3,3.8))
for j,(_,r) in enumerate(d.iterrows()):
    ax.errorbar(r.difference,j,
     xerr=[[r.difference-r.ci_lower],[r.ci_upper-r.difference]],
     fmt="o",color=BLUE,capsize=3,ms=6,lw=1.6)
ax.set_yticks(range(4),["Equal fireline units","Equal source records",
                      "Equal fires; fireline units within fires","Equal fires; source records within fires"])
ax.set(ylim=(3.6,-.6),xlim=(-.045,.02),xlabel="LightGBM Brier score − matched historical baseline Brier score")
ax.axvline(0,color=GRAY,lw=1,ls="--")
ax.set_title("Forward-year conclusions depend on the averaging objective",loc="left",fontsize=12)
clean(ax,"x")
fig.tight_layout(rect=[0,.13,1,1])
fig.text(.02,.075,"2021–2024 tests. Negative: LightGBM has lower error. Positive: the constant baseline has lower error.",fontsize=10)
fig.text(.02,.025,"Each baseline uses preceding years only. Bars: paired 95% intervals from 2,000 whole-fire resamples.",fontsize=10)
save(fig,"supp_baseline_weighting_sensitivity")


## Supplement candidate: nearby cutoffs on a common set of fires

The same eligible fires are used across cutoffs within each family. Every comparison is the Held fraction inside the condition group minus the fraction among all remaining eligible fireline units in the same fire (not only the same construction type), then averaged equally across fires, with source-record weights within fires. These are not the unit-weighted group rates in the main figure. All 13 planned cutoff combinations are shown. Orange squares mark the original tree cutoffs; no cutoff was selected as a test-data winner.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10.3,5.5),sharex=True)
for ax,family,title in zip(axes,["humid_not_dozer","near_dozer"],
                         ["Other recorded types: humidity and proximity","Completed Dozer Line: proximity"]):
    d=cutoffs.loc[(cutoffs.scope=="2023_2024")&(cutoffs.weight=="fire_source")&(cutoffs.family==family)].copy()
    assert d.common_fires.nunique()==1
    labs=[]
    for j,(_,r) in enumerate(d.iterrows()):
        if family=="humid_not_dozer":
            rh=float(r.group.split("__")[0].removeprefix("rh_"))
            dist=float(r.group.split("__")[1].removeprefix("distance_"))
            labs.append(f"RH > {rh:.1f}%; D ≤ {dist:.1f} m")
        else:
            dist=float(r.group.removeprefix("dozer_distance_"))
            labs.append(f"D ≤ {dist:.1f} m")
        color=ORANGE if bool(r.original_reference) else BLUE
        ax.errorbar(100*r.within_fire_difference,j,
          xerr=[[100*(r.within_fire_difference-r.ci_lower)],[100*(r.ci_upper-r.within_fire_difference)]],
          fmt="s" if bool(r.original_reference) else "o",color=color,capsize=2,ms=5,lw=1.4)
    ax.set_yticks(range(len(d)),labs)
    ax.set(ylim=(len(d)-.5,-.5),xlim=(-43,15),
           xlabel="Within-fire Held difference\n(percentage points)")
    ax.axvline(0,color=GRAY,ls="--",lw=1)
    ax.set_title(title+f"\n{int(d.common_fires.iloc[0])} common fires",fontsize=10)
    clean(ax,"x")
fig.suptitle("Nearby-cutoff sensitivity, 2023–2024",fontsize=13)
fig.tight_layout(rect=[0,.19,1,.91],w_pad=2.5)
fig.text(.02,.125,"RH: forecast mean relative humidity, 0–18 h. D: closest mapped fire-area-to-fireline distance.",fontsize=10)
fig.text(.02,.08,"Equal-fire contrasts use source-record weights within fires. Orange squares: original cutoffs.",fontsize=10)
fig.text(.02,.035,"Bars: pointwise 95% intervals from 2,000 fire resamples; exploratory checks, not operational thresholds.",fontsize=10)
save(fig,"supp_condition_cutoff_stability")


## Provenance, captions, and preservation

The manifest records source hashes and exported files. Proposed captions are supplied as plain text, not inserted LaTeX. The current Overleaf paper will continue showing its old figures until we edit the corresponding figure blocks. Annual error point estimates are not evidence that all annual differences have intervals excluding zero. In particular, the 2021 unit-weighted difference interval includes zero. Main condition rates are observed frequencies, not fitted probabilities.


In [ ]:
captions={
"forward_year_unit_performance":
"Forward-year LightGBM performance. Training uses 2020 through the year preceding each test. (a) Pooled ROC-AUC compares fireline units across the test dataset; within-fire ROC-AUC is calculated in fires containing both outcomes and then averaged equally across fires. The dotted line denotes chance ordering. (b) Pooled Brier score gives every test fireline unit equal weight; lower values indicate lower probability error. The historical baseline assigns each test fireline unit the corresponding training-data Held fraction. Curves show annual point estimates; uncertainty for differences is reported separately.",
"condition_groups_unit_held_rates":
"Recorded Held fractions in eight condition groups learned by a depth-three tree from 2020–2022 outcomes and applied unchanged to 2023–2024 fireline units. The tree was trained with equal-fire and equal-source-record weights; the displayed outcome fractions give fireline units equal weight. Cutoffs are rounded for display. RH is mean forecast relative humidity over 0–18 h after prediction time, gust is forecast at +18 h, E is mean elevation, and D is the closest distance from a fireline unit to the mapped fire area at prediction time, not a distance applying to the full fireline length. Other recorded types exclude Completed Dozer Line. Counts show fireline units and represented fires; fires can contribute to multiple groups. Bars are pointwise 95% intervals from 2,000 whole-fire bootstrap resamples. The analysis is exploratory following prior examination of the archive, not independent confirmation. The groups describe historical associations, not construction prescriptions.",
"supp_condition_groups_by_year":
"Recorded Held fractions for the same eight fixed condition groups in each test year and combined. Group order matches the main condition figure. Each fireline unit receives equal weight. Counts indicate fireline units; the open point marks fewer than 200 fireline units, 50 source records, or 10 fires. The near-dozer group in 2023 has 133 fireline units. Bars are pointwise 95% intervals from 2,000 whole-fire resamples.",
"supp_baseline_weighting_sensitivity":
"Forward-year LightGBM probability error relative to historical constant predictions under four averaging choices. Training-baseline weights match test-scoring weights within each row. Source weighting divides each source record's weight among its retained fireline units; fire weighting gives every fire equal total weight. Negative differences favor LightGBM. Bars are paired pointwise 95% intervals from 2,000 complete-fire resamples with predictions fixed. All baselines use preceding years only.",
"supp_condition_cutoff_stability":
"Sensitivity of two condition associations to all 13 fixed nearby cutoff settings in the combined 2023–2024 data. Differences compare Held fractions inside each group with all remaining eligible fireline units within the same fire, not only fireline units of the same construction type; source-record weights are applied within fires and contrasts are averaged equally across fires. Comparisons use the same 56 fires across humidity/proximity settings and the same 40 fires across dozer proximity settings. Orange squares mark original tree cutoffs. Bars are pointwise 95% intervals from 2,000 fire resamples. Annual and alternative-model checks remain necessary; these combined exploratory intervals are not independent confirmations.",
}
readme = ["FIGURE ADDITIONS ONLY — NO MANUSCRIPT SOURCE HAS BEEN EDITED.",
"All original files, including paper.tex, supplement.tex, tables and existing figures, are preserved.",
"New figures are not automatically included. Insert them only while revising the corresponding prose and captions.",
"MAIN CANDIDATES: forward_year_unit_performance; condition_groups_unit_held_rates.",
"SUPPLEMENT CANDIDATES: supp_condition_groups_by_year; supp_baseline_weighting_sensitivity; supp_condition_cutoff_stability.",
"PDF files are vector graphics; PNG copies are 300-dpi previews.",
"Do not describe 08D's 2,000-resample intervals as 5,000-resample intervals.",
"Do not relabel executed fire-weighted model selection or condition-tree training as unit-weighted.",
"Do not mix observed group Held fractions with fitted probabilities.",
"Source CSV copies and a provenance manifest accompany the figures. The notebook rebuilds from the unchanged project outputs.",
"Paper and supplement pagination/compilation have not been changed or newly verified.",
"", "PROPOSED CAPTIONS (for paragraph-by-paragraph review):"]
for name,caption in captions.items():
    readme.extend(["",name,caption])
(OUT/"READ_ME_FIRST.txt").write_text("\n".join(readme)+"\n",encoding="utf-8")
manifest={
 "purpose":"figure additions, no manuscript edits",
 "production_run":RUN.name,
 "notebook":"notebooks/08_series_paper_figure_refresh.ipynb",
 "paper_sha256":original_paper_hash,
 "sources":{name:{"path":str(path),"sha256":hashlib.sha256(path.read_bytes()).hexdigest()} for name,path in sources.items()},
 "figures":[{"path":str(path.relative_to(PROJECT)),"sha256":hashlib.sha256(path.read_bytes()).hexdigest()} for path in generated],
 "captions":captions,
 "bootstrap_replicates":2000,
 "display_group_to_original_leaf":{str(j+1):leaf for j,leaf in enumerate(order)},
 "no_training_performed":True,
}
assert hashlib.sha256((PROJECT/"paper.tex").read_bytes()).hexdigest()==original_paper_hash
(OUT/"figure_provenance.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False)+"\n",encoding="utf-8")
print("Created",len(generated)//2,"figure pairs in",OUT)
print("paper.tex is byte-for-byte unchanged.")
